# OBJ1 数据准备 + OBJ2 Baseline（TF‑IDF + Logistic Regression）

本 notebook 面向 Sentiment140（training.1600000.processed.noemoticon.csv），完成：

- **OBJ1**：清洗/预处理 + **60/10/30**（train/val/test）分层划分
- **OBJ2**：建立 **baseline supervised sentiment classifier（TF‑IDF + Logistic Regression）**

> 备注：保留否定词、感叹号/问号、重复字母、hashtag 等情感信号；避免过度清洗导致信息损失。

In [1]:
# !pip -q install scikit-learn pandas numpy joblib


## 1. 读取数据（Sentiment140）

原始字段：`target, ids, date, flag, user, text`。标准 Sentiment140 通常只有 **0(neg)** 与 **4(pos)**。

In [3]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../Dataset/training.1600000.processed.noemoticon.csv")

COLS = ["target","ids","date","flag","user","text"]

df = pd.read_csv(
    DATA_PATH,
    encoding="ISO-8859-1",
    header=None,
    names=COLS,
    usecols=["target","text"],  # baseline 只用标签与文本
    dtype={"target":"int32","text":"string"},
)

# 只保留 0/4（二分类）
df = df[df["target"].isin([0,4])].copy()
df["label"] = (df["target"] == 4).astype("int8")  # 1=pos, 0=neg

print(df.shape)
df.head()


(1600000, 3)


,target,text,label
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0
1,0,is upset that he can't update his Facebook by ...,0
2,0,@Kenichan I dived many times for the ball. Man...,0
3,0,my whole body feels itchy and like its on fire,0
4,0,"@nationwideclass no, it's not behaving at all....",0


## 2. 轻量但“情感友好”的预处理

原则：
- ✅ 保留否定信息：**不删除 stopwords**（或至少保留 not/no/never/n't）
- ✅ 保留强度信号：`! ?`、重复字符（soooo）、全部大写的强调
- ✅ 保留 hashtag 语义（#happy）
- ✅ 只移除低价值噪声：URL、@提及、HTML、冗余空白

> 这里采用“最小损失清洗”，把复杂特征交给 TF‑IDF（含 bigram）学习。

In [4]:
import re

URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
USER_RE = re.compile(r"@\w+")
HTML_RE = re.compile(r"&\w+;")
# 仅移除除常见情感符号外的杂字符：保留 ! ? # '（用于n't）
BAD_CHARS_RE = re.compile(r"[^0-9A-Za-z\s!?#!'’]")

MULTISPACE_RE = re.compile(r"\s+")
REPEAT_RE = re.compile(r"(.)\1{3,}")  # 超过3次重复归一

def normalize_repeats(text: str) -> str:
    # soooo -> soo（保留一定强度信息）
    return REPEAT_RE.sub(r"\1\1", text)

def preprocess(text: str) -> str:
    if text is None:
        return ""
    t = str(text)
    t = HTML_RE.sub(" ", t)
    t = URL_RE.sub(" <URL> ", t)
    t = USER_RE.sub(" <USER> ", t)
    t = normalize_repeats(t)
    # 保留 hashtag：#happy -> HASHTAG_happy（让 tfidf 能吃到）
    t = re.sub(r"#(\w+)", r" HASHTAG_\1 ", t)
    # 统一引号，保留 n't
    t = t.replace("’", "'")
    t = BAD_CHARS_RE.sub(" ", t)
    t = MULTISPACE_RE.sub(" ", t).strip()
    return t.lower()

df["text_clean"] = df["text"].map(preprocess)

df[["text","text_clean","label"]].head(10)


,text,text_clean,label
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",user url awww that's a bummer you shoulda got ...,0
1,is upset that he can't update his Facebook by ...,is upset that he can't update his facebook by ...,0
2,@Kenichan I dived many times for the ball. Man...,user i dived many times for the ball managed t...,0
3,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire,0
4,"@nationwideclass no, it's not behaving at all....",user no it's not behaving at all i'm mad why a...,0
5,@Kwesidei not the whole crew,user not the whole crew,0
6,Need a hug,need a hug,0
7,@LOLTrish hey long time no see! Yes.. Rains a...,user hey long time no see! yes rains a bit onl...,0
8,@Tatiana_K nope they didn't have it,user nope they didn't have it,0
9,@twittera que me muera ?,user que me muera ?,0


## 3. 分层划分：60/10/30（train/val/test）

按照 Interim Report 的 OBJ1 要求进行固定比例划分，并使用 `stratify` 保持正负样本比例一致。

In [5]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# 先切出 train 60% 与 temp 40%
train_df, temp_df = train_test_split(
    df[["text_clean","label"]],
    test_size=0.40,
    random_state=RANDOM_STATE,
    stratify=df["label"],
)

# 再把 temp 40% 切成 val 10% 与 test 30%
# val 在总量占 10% => 在 temp 中占 10/40 = 0.25
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.75,
    random_state=RANDOM_STATE,
    stratify=temp_df["label"],
)

def dist(name, d):
    return name, len(d), float(d["label"].mean())

for item in [dist("train", train_df), dist("val", val_df), dist("test", test_df)]:
    print(item)


('train', 960000, 0.5)
('val', 160000, 0.5)
('test', 480000, 0.5)


## 4. OBJ2 Baseline：TF‑IDF + Logistic Regression

推荐设置：
- `ngram_range=(1,2)`：捕捉 **not good** 这类否定 bigram
- `sublinear_tf=True`：减少超高频词的影响
- `min_df`：过滤极低频噪声
- `class_weight='balanced'`：以防比例不完全平衡

> 注意：baseline 不做 embedding、不用深度网络，符合 OBJ2 要求。

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf_lr = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(
        max_iter=200,
        solver="saga",
        n_jobs=-1,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ))
])

X_train, y_train = train_df["text_clean"], train_df["label"]
X_val, y_val     = val_df["text_clean"],   val_df["label"]
X_test, y_test   = test_df["text_clean"],  test_df["label"]

tfidf_lr.fit(X_train, y_train)

print("val_acc:", tfidf_lr.score(X_val, y_val))
print("test_acc:", tfidf_lr.score(X_test, y_test))


val_acc: 0.8224375
test_acc: 0.8225604166666667


## 5. 评估：Accuracy + Precision/Recall/F1 + Confusion Matrix

OBJ4 会更系统，但这里先给 baseline 的核心评估结果。

In [7]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = tfidf_lr.predict(X_test)

print(classification_report(y_test, y_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0     0.8267    0.8162    0.8214    240000
           1     0.8185    0.8290    0.8237    240000

    accuracy                         0.8226    480000
   macro avg     0.8226    0.8226    0.8226    480000
weighted avg     0.8226    0.8226    0.8226    480000

Confusion matrix:
 [[195876  44124]
 [ 41047 198953]]


## 6. 保存产物（给 GUI/后续对比复用）

保存：
- `baseline_tfidf_lr.joblib`：整个 pipeline（含向量化器+模型）
- `splits/*.csv`：三份划分（便于复现实验）

In [8]:
import joblib
from pathlib import Path

OUT_DIR = Path("artifacts")
OUT_DIR.mkdir(exist_ok=True)

# === Model artifact (binary classifier) ===
MODEL_PATH = OUT_DIR / "baseline_tfidf_lr_binary.joblib"
joblib.dump(tfidf_lr, MODEL_PATH)

# === Fixed data splits (OBJ1) ===
SPLIT_DIR = OUT_DIR / "splits"
SPLIT_DIR.mkdir(exist_ok=True)

train_df.to_csv(SPLIT_DIR / "train_60.csv", index=False)
val_df.to_csv(SPLIT_DIR / "val_10.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test_30.csv", index=False)

print("Saved model to:", MODEL_PATH.resolve())
print("Saved splits to:", SPLIT_DIR.resolve())


Saved to: /Users/victor/Desktop/Graduation_Thesis/Code/artifacts


## 7. 快速推理：加入 Neutral（中性）标签，并输出用于 Happiness Index 的分数

由于 Sentiment140 训练集本身是二分类（0=NEG, 4=POS），我们仍然用二分类模型训练（符合 OBJ2 的传统 baseline）。

根据老师关于“引入 Neutral”的建议，这里采用 **post-hoc thresholding**：

- `P(pos) >= POS_TH` → **POSITIVE**
- `P(pos) <= NEG_TH` → **NEGATIVE**
- 其余区间 → **NEUTRAL**（模型不确定区间）

这样不会改变训练方式，但能在推理/指数构建阶段得到三类输出，便于 GUI 展示与 Happiness Index 聚合。


In [9]:
import numpy as np

# --- Neutral 阈值（可在实验中做敏感性分析）---
NEG_TH = 0.40
POS_TH = 0.60

def proba_to_3class(proba_pos: np.ndarray, neg_th: float = NEG_TH, pos_th: float = POS_TH):
    """将二分类概率 P(pos) 映射为三类标签：NEGATIVE / NEUTRAL / POSITIVE。
    注意：这是 post-hoc 阈值策略，不是三分类训练。
    """
    labels = np.full(shape=proba_pos.shape, fill_value='NEUTRAL', dtype=object)
    labels[proba_pos <= neg_th] = 'NEGATIVE'
    labels[proba_pos >= pos_th] = 'POSITIVE'
    return labels

def predict_sentiment(texts, neg_th: float = NEG_TH, pos_th: float = POS_TH):
    """返回：[(text, P(pos), binary_label, three_class_label), ...]"""
    texts_clean = [preprocess(t) for t in texts]
    proba_pos = tfidf_lr.predict_proba(texts_clean)[:, 1]
    binary_label = (proba_pos >= 0.5).astype(int)
    three_class_label = proba_to_3class(proba_pos, neg_th=neg_th, pos_th=pos_th)
    return list(zip(texts, proba_pos, binary_label, three_class_label))

# demo
examples = [
    'I am sooooo happy!!!',
    'not good at all...',
    'This is fine? maybe.',
]
predict_sentiment(examples)


[('I am sooooo happy!!!', np.float64(0.9389), np.int64(1)),
 ('not good at all...', np.float64(0.0098), np.int64(0)),
 ('This is fine? maybe.', np.float64(0.8595), np.int64(1))]

In [ ]:
import json
import pandas as pd

# === Save Neutral threshold policy (post-hoc) ===
THRESHOLD_PATH = OUT_DIR / "neutral_thresholds.json"
with open(THRESHOLD_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "neg_th": float(NEG_TH),
            "pos_th": float(POS_TH),
            "policy": "post_hoc_thresholding_on_p_positive",
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

# === Export test predictions for later analysis / plots / thesis tables ===
# binary (for main metrics)
x_test_clean = test_df["text_clean"].tolist()
proba_pos_test = tfidf_lr.predict_proba(x_test_clean)[:, 1]
pred_binary_test = (proba_pos_test >= 0.5).astype(int)

pred_binary_df = pd.DataFrame(
    {
        "text": test_df["text"].tolist(),
        "text_clean": test_df["text_clean"].tolist(),
        "y_true": test_df["label"].astype(int).tolist(),
        "proba_pos": proba_pos_test,
        "pred_binary": pred_binary_test,
    }
)
pred_binary_path = OUT_DIR / "pred_test_binary.csv"
pred_binary_df.to_csv(pred_binary_path, index=False)

# 3-class mapping (for Neutral strategy analysis & Happiness Index)
pred_3class_test = proba_to_3class(proba_pos_test, neg_th=NEG_TH, pos_th=POS_TH)

pred_neutral_df = pred_binary_df.copy()
pred_neutral_df["pred_3class"] = pred_3class_test
pred_neutral_path = OUT_DIR / f"pred_test_neutral_{NEG_TH:.2f}_{POS_TH:.2f}.csv"
pred_neutral_df.to_csv(pred_neutral_path, index=False)

print("Saved Neutral thresholds to:", THRESHOLD_PATH.resolve())
print("Saved binary predictions to:", pred_binary_path.resolve())
print("Saved Neutral predictions to:", pred_neutral_path.resolve())


In [ ]:
# --- Happiness Index（示例）：把三类标签聚合成一个分数 ---
# 一种简单做法：POS=+1, NEU=0, NEG=-1
def happiness_score_from_3class(labels_3):
    mapping = {'POSITIVE': 1, 'NEUTRAL': 0, 'NEGATIVE': -1}
    vals = np.array([mapping[l] for l in labels_3], dtype=float)
    return vals.mean() if len(vals) else float('nan')

preds = predict_sentiment(examples)
labels3 = [p[3] for p in preds]
labels3, happiness_score_from_3class(labels3)
